# Voter flows 2022 → 2026 (Knesset 25 → 26)

Recreates the LittleMoiz flow chart ([x.com/LittleMoiz/status/2118240810411986997](https://x.com/LittleMoiz/status/2118240810411986997/photo/1))
with this project's approach: polling places aligned by city + box number, abstained = registered − valid,
Bayesian ecological inference (column-softmax transition matrix with loyalty bias, Dirichlet-multinomial).
Model fit: `python kn25_26_flows/flows.py` (conda env `bayesian`). Flows are scaled by each group's **national** 2022 size, as in the original.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "kn25_26_flows" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "kn25_26_flows")]

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from flows import FROM, GROUPS_25, GROUPS_26, OUT, TO, load_groups
from src.plot_utils import heb
from src.sankey import sankey

idata = az.from_netcdf(OUT / "trace.nc")
x1 = pd.read_parquet(OUT / "x1.parquet")
print(f"{len(x1):,} aligned polling places, {x1.values.sum():,.0f} of "
      f"{load_groups(25, GROUPS_25)[FROM].values.sum():,.0f} voters incl. double envelopes (2022)")

4,117 aligned polling places, 3,186,435 of 7,247,518 voters incl. double envelopes (2022)


## Diagnostics

In [2]:
print(az.summary(idata, var_names=["diag_bias"]))
s = az.summary(idata, var_names=["M"])
print("max r_hat:", s["r_hat"].max(), " min ess_bulk:", s["ess_bulk"].min())
print("divergences:", int(idata.sample_stats["diverging"].sum()))

            mean   sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
diag_bias  4.592  0.3   4.036    5.156      0.008    0.005    1380.0   

           ess_tail  r_hat  
diag_bias    1807.0    1.0  
max r_hat: 1.0  min ess_bulk: 2430.0
divergences: 0


## Transition matrix (share of each 2022 group, columns sum to 1)

In [3]:
M = idata.posterior["M"].stack(s=("chain", "draw")).values  # to x from x samples
share = pd.DataFrame(M.mean(-1), index=TO, columns=FROM)
share.round(2)

,likud,shas,rz_otzma,bayit_yehudi,national_unity,change,utj,arab,other,abstained
likud,0.92,0.08,0.00,0.00,0.00,0.00,0.00,0.0,0.01,0.01
shas,0.00,0.72,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00
rz_otzma,0.00,0.10,0.95,0.19,0.00,0.01,0.00,0.0,0.06,0.00
winter,0.02,0.00,0.02,0.56,0.04,0.01,0.00,0.0,0.07,0.00
hendel,0.06,0.00,0.00,0.12,0.08,0.01,0.00,0.0,0.35,0.00
change,0.00,0.00,0.00,0.10,0.87,0.97,0.00,0.0,0.48,0.06
utj,0.00,0.01,0.00,0.00,0.00,0.00,0.95,0.0,0.00,0.00
arab,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.0,0.00,0.01
other,0.00,0.01,0.02,0.01,0.00,0.00,0.04,0.0,0.01,0.01
abstained,0.00,0.06,0.00,0.01,0.00,0.00,0.01,0.0,0.00,0.89


## Flows in thousands of voters (scaled by national 2022 group size), 90% credible interval

In [4]:
n25 = load_groups(25, GROUPS_25)[FROM].sum()
flows = M * n25.values[None, :, None]
lo, mean, hi = (np.quantile(flows, 0.05, -1), flows.mean(-1), np.quantile(flows, 0.95, -1))
flow_k = pd.DataFrame(mean / 1e3, index=TO, columns=FROM)
print("2022 group sizes (k):", (n25 / 1e3).round().astype(int).to_dict())
print("2026 totals: model vs actual (k)")
actual26 = load_groups(26, GROUPS_26)[TO].sum() / 1e3
print(pd.DataFrame({"model": flow_k.sum(1), "actual": actual26}).round())
flow_k.round().astype(int)

2022 group sizes (k): {'likud': 1115, 'shas': 393, 'rz_otzma': 516, 'bayit_yehudi': 57, 'national_unity': 432, 'change': 1388, 'utj': 280, 'arab': 511, 'other': 71, 'abstained': 2483}
2026 totals: model vs actual (k)
            model  actual
likud      1098.0  1168.0
shas        292.0   300.0
rz_otzma    571.0   617.0
winter       99.0    96.0
hendel      151.0   155.0
change     1927.0  2149.0
utj         275.0   313.0
arab        537.0   843.0
other        57.0    51.0
abstained  2240.0  2091.0


,likud,shas,rz_otzma,bayit_yehudi,national_unity,change,utj,arab,other,abstained
likud,1026,33,1,0,1,0,0,0,1,36
shas,0,282,0,0,0,0,1,0,0,7
rz_otzma,1,40,493,11,2,12,0,0,4,8
winter,24,1,12,32,16,8,0,0,5,3
hendel,63,1,1,7,36,18,0,0,25,1
change,1,0,0,6,377,1349,0,0,34,159
utj,0,5,0,0,0,0,265,0,0,3
arab,0,0,0,0,0,1,0,511,0,24
other,0,5,9,0,0,0,11,0,1,31
abstained,0,26,0,0,0,0,2,0,0,2210


## Comparison with the LittleMoiz chart

Their "did not vote / small list" source ≈ our `abstained` + `other` (2022).

In [5]:
fi = lambda t, f: (TO.index(t), FROM.index(f))
def k(t, fs):
    ii = [fi(t, f) for f in fs]
    tot = sum(flows[i] for i in ii)
    return f"{tot.mean()/1e3:.0f} [{np.quantile(tot, .05)/1e3:.0f}–{np.quantile(tot, .95)/1e3:.0f}]"
rows = [
    ("Shas stayed (%)", "73%", f"{share.loc['shas','shas']:.0%}"),
    ("Likud stayed (%)", "92%", f"{share.loc['likud','likud']:.0%}"),
    ("Shas → RZ+Otzma", 49, k("rz_otzma", ["shas"])),
    ("Shas → Likud", 34, k("likud", ["shas"])),
    ("Likud → RZ+Otzma", 91, k("rz_otzma", ["likud"])),
    ("RZ+Otzma → Winter", 37, k("winter", ["rz_otzma"])),
    ("RZ+Otzma → change bloc", 21, k("change", ["rz_otzma"])),
    ("RZ+Otzma → Hendel", 22, k("hendel", ["rz_otzma"])),
    ("Bayit Yehudi → Winter", 13, k("winter", ["bayit_yehudi"])),
    ("Bayit Yehudi → change bloc", 8, k("change", ["bayit_yehudi"])),
    ("RZ+Otzma → Likud", 28, k("likud", ["rz_otzma"])),
    ("National Unity → Hendel", 32, k("hendel", ["national_unity"])),
    ("non-voters/small → change bloc", 140, k("change", ["abstained", "other"])),
    ("non-voters/small → Hendel", 67, k("hendel", ["abstained", "other"])),
    ("non-voters/small → Likud", 53, k("likud", ["abstained", "other"])),
    ("non-voters/small → RZ+Otzma", 16, k("rz_otzma", ["abstained", "other"])),
    ("non-voters/small → Winter", 13, k("winter", ["abstained", "other"])),
]
pd.DataFrame(rows, columns=["flow", "LittleMoiz (k)", "ours: mean [90% CI] (k)"])

,flow,LittleMoiz (k),ours: mean [90% CI] (k)
0,Shas stayed (%),73%,72%
1,Likud stayed (%),92%,92%
2,Shas → RZ+Otzma,49,40 [36–44]
3,Shas → Likud,34,33 [29–37]
4,Likud → RZ+Otzma,91,1 [0–2]
5,RZ+Otzma → Winter,37,12 [9–14]
6,RZ+Otzma → change bloc,21,0 [0–1]
7,RZ+Otzma → Hendel,22,1 [0–2]
8,Bayit Yehudi → Winter,13,32 [30–34]
9,Bayit Yehudi → change bloc,8,6 [3–8]


## Flow chart

In [6]:
H25 = {"likud": "הליכוד", "shas": "ש״ס", "rz_otzma": "הציונות הדתית ועוצמה", "bayit_yehudi": "הבית היהודי",
       "national_unity": "המחנה הממלכתי", "change": "יש עתיד, ליברמן, עבודה, מרצ", "utj": "יהדות התורה",
       "arab": "רשימות ערביות", "other": "רשימות קטנות", "abstained": "לא הצביעו"}
H26 = {"likud": "הליכוד", "shas": "ש״ס", "rz_otzma": "הציונות הדתית + עוצמה", "winter": "וינטר", "hendel": "הנדל וזליכה",
       "change": "גוש השינוי", "utj": "יהדות התורה", "arab": "רשימות ערביות", "other": "רשימות קטנות", "abstained": "לא הצביעו"}
long = (flow_k.stack().rename("vote_count").rename_axis(["to", "from"]).reset_index())
long = long[(long["vote_count"] >= 10) & ~((long["from"] == "abstained") & (long["to"] == "abstained"))]
long["from_category"] = long["from"].map(lambda s: heb(H25[s] + " 2022"))
long["to_category"] = long["to"].map(lambda s: heb(H26[s] + " 2026"))
palette = dict(likud="#1f4e9c", shas="#2a2a2a", rz_otzma="#e08a00", bayit_yehudi="#c2a14d", national_unity="#4a90c2",
               change="#c8102e", utj="#555555", arab="#2e8b57", other="#bbbbbb", abstained="#dddddd")
colors = {heb(H25[g] + " 2022"): palette[g] for g in FROM} | {heb(H26[g] + " 2026"): palette.get(g, "#7a3fb0") for g in TO}
colors[heb(H26["winter"] + " 2026")] = "#9c5a00"; colors[heb(H26["hendel"] + " 2026")] = "#7a3fb0"
plt.rcParams["font.serif"] = ["Arial"]  # sankey() forces serif; Arial has Hebrew
fig, ax = plt.subplots(figsize=(10, 14))
sankey(long, category_colors=colors, left_labels=[heb(H25[g] + " 2022") for g in FROM if heb(H25[g] + " 2022") in set(long["from_category"])],
       right_labels=[heb(H26[g] + " 2026") for g in TO if heb(H26[g] + " 2026") in set(long["to_category"])], font_size=11, ax=ax)
ax.set_title(heb("איך זזו הקולות, 2022 עד 2026 (זרמים ≥10 אלף; ללא נשארו-בבית)"), fontsize=14)
fig.savefig(OUT / "flows_2022_2026.png", dpi=150, bbox_inches="tight")